# 4 - Alignment Score Behavioral Checks

**Covers:** Section 4.9.3(D) in full, the engineered-input correctness check from Section 4.9.1, and the Table 1 dataset profile.

Run this **before** the split and the model ladder. Every check here is cheap, and together they forecast what Section 4.9.3(A) will conclude. Section 4.9.3(D) says so explicitly: a gap distribution concentrated near zero *"would itself explain a null predictive result and is worth establishing before interpreting one."*

**Produces:** `artifacts/behavioral_checks.json`, `artifacts/fig_gap_distribution.png`, `artifacts/fig_gap_by_genre.png`.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))
import modeling_config as mc

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

df = mc.load_corpus()
print('rows:', len(df))
print('columns:', list(df.columns))

## A. Corpus profile (Table 1) and the Section 4.2 exclusion criteria

Also prints the RMSE band that is actually consistent with Table 11's Control R2 of 0.05-0.20, since a mean-predictor's RMSE is exactly the target's standard deviation.

In [ ]:
counts = df['genre'].value_counts()
sigma = float(df['popularity'].std())

profile = {
    'retained_tracks': int(len(df)),
    'unique_track_ids': int(df['id'].nunique()),
    'year_min': int(df['year'].min()),
    'year_max': int(df['year'].max()),
    'popularity_mean': float(df['popularity'].mean()),
    'popularity_median': float(df['popularity'].median()),
    'popularity_std': sigma,
    'popularity_min': int(df['popularity'].min()),
    'popularity_max': int(df['popularity'].max()),
    'n_genres': int(df['genre'].nunique()),
    'genre_imbalance_ratio': float(counts.iloc[0] / counts.iloc[-1]),
}
print(json.dumps(profile, indent=2))
print()
print(counts.to_string())

# Section 4.2 says these hold on the retained corpus. Assert rather than assume.
assert (df['popularity'] > 0).all(), 'zero-popularity rows present'
assert df['is_valid_lyrics'].all(), 'invalid-lyric rows present'
missing = df[mc.AUDIO_12 + mc.LYRIC_COLS + mc.ALIGNMENT_COLS].isna().sum()
assert missing.sum() == 0, missing[missing > 0]
assert df['id'].is_unique, 'duplicate track ids'
print('\nSection 4.2 exclusion criteria hold; no missing descriptor / lyric / alignment values.')

print('\n-- Table 11 cross-check --')
print('Baseline (mean-predictor) RMSE is the target sigma: %.2f popularity points' % sigma)
print('Control RMSE consistent with R2 in [0.05, 0.20]: %.2f - %.2f'
      % (sigma * np.sqrt(1 - 0.20), sigma * np.sqrt(1 - 0.05)))
print('Table 11 currently states 15 - 20. An RMSE of 20 would imply R2 = %.2f.'
      % (1 - (20.0 / sigma) ** 2))

## B. Correctness of the engineered input (Section 4.9.1)

Hand-computed reference cases spanning the range, then a bit-for-bit recomputation across the whole corpus. Section 4.9.1: *"If the feature is non-deterministic or misimplemented, any observed difference between the two models reflects computation variance rather than the construct being tested."*

In [ ]:
cases = pd.DataFrame([
    {'case': 'max positive lyrics vs max negative audio', 'valence': 0.00, 'lyric_sentiment':  1.0, 'expected':  2.0},
    {'case': 'max negative lyrics vs max positive audio', 'valence': 1.00, 'lyric_sentiment': -1.0, 'expected': -2.0},
    {'case': 'neutral lyrics vs neutral audio',           'valence': 0.50, 'lyric_sentiment':  0.0, 'expected':  0.0},
    {'case': 'mild positive lyrics vs sad-sounding audio','valence': 0.25, 'lyric_sentiment':  0.4, 'expected':  0.9},
    {'case': 'bleak lyrics vs upbeat audio',              'valence': 0.80, 'lyric_sentiment': -0.6, 'expected': -1.2},
])
cases['valence_normalized'] = 2 * cases['valence'] - 1
cases['computed'] = cases['lyric_sentiment'] - cases['valence_normalized']
cases['matches'] = np.isclose(cases['computed'], cases['expected'])
print(cases.to_string(index=False))
assert cases['matches'].all(), 'a hand-computed reference case failed'

# corpus-wide determinism
recomputed = df['lyric_sentiment'] - (2 * df['valence'] - 1)
deviation = float((df['alignment_gap'] - recomputed).abs().max())
print('\nmax |stored - recomputed| across the corpus: %.3e' % deviation)
assert deviation < 1e-9, 'stored alignment_gap disagrees with the Section 4.2.1 formula'
assert df['alignment_gap'].between(-2, 2).all(), 'alignment_gap outside [-2, 2]'
print('bounded in [-2, 2]: observed range (%.4f, %.4f)'
      % (df['alignment_gap'].min(), df['alignment_gap'].max()))
print('sign convention: positive = lyrics read brighter than the music sounds.')

## C. Distribution across the corpus, against Attia (2025)

Attia reports 46% agreement from a **cluster-based binary**, not a threshold on this gap, so there is no exact correspondence. The honest comparison is to report agreement share at several tolerances and say which one lands nearest 46%.

In [ ]:
gap = df['alignment_gap']
print('mean %.3f   median %.3f   std %.3f' % (gap.mean(), gap.median(), gap.std()))
print('share positive (lyrics brighter than music): %.1f%%' % (100 * (gap > 0).mean()))
print('share negative (music brighter than lyrics): %.1f%%' % (100 * (gap < 0).mean()))
print()

tolerances = [0.10, 0.25, 0.50, 0.75, 1.00]
tol = pd.DataFrame([
    {'tau': t,
     'agreement_pct': 100 * (gap.abs() <= t).mean(),
     'mismatch_pct': 100 * (gap.abs() > t).mean()}
    for t in tolerances
])
print(tol.round(1).to_string(index=False))

nearest = tol.iloc[(tol['agreement_pct'] - 46.0).abs().argmin()]
print('\ntau closest to Attia 46%% agreement: tau = %.2f (%.1f%% agreement)'
      % (nearest['tau'], nearest['agreement_pct']))
print('A distribution tightly concentrated near zero would predict a null')
print('result (Section 4.9.3 D). Check the spread above before reading 4.9.3(A).')

## D. Distribution by genre

Section 4.9.3(D): Attia found mismatch from 55% (hip-hop) to 92% (jazz). *"Reproducing the genre-patterned structure Attia observed would provide independent corroboration that the formula is measuring the same phenomenon."* Compare the **ordering**, not the absolute levels.

In [ ]:
TAU = 0.50  # headline tolerance; vary it above and re-read this table

grouped = df.groupby('genre')['alignment_gap']
by_genre = pd.DataFrame({
    'n': grouped.size(),
    'mean_gap': grouped.mean(),
    'median_gap': grouped.median(),
    'std_gap': grouped.std(),
    'mismatch_pct': grouped.apply(lambda s: 100 * (s.abs() > TAU).mean()),
}).sort_values('mismatch_pct', ascending=False)
print('mismatch = |alignment_gap| > %.2f\n' % TAU)
print(by_genre.round(2).to_string())
print('\nAttia (2025) reference points: jazz 92%% mismatch, hip-hop 55%%.')
print('Rank of Jazz here:   ', list(by_genre.index).index('Jazz') + 1, 'of', len(by_genre))
print('Rank of Hip-Hop here:', list(by_genre.index).index('Hip-Hop') + 1, 'of', len(by_genre))

## E. Manual inspection of both tails

Section 4.9.3(D): negative extremes should surface upbeat, major-key production against bleak lyrics; positive extremes the reverse. This is the qualitative check that the formula's tails correspond to the phenomenon the study claims to measure.

In [ ]:
cols = ['name', 'artists', 'genre', 'valence', 'mode', 'lyric_sentiment', 'alignment_gap', 'popularity']

print('=== MOST POSITIVE gap: lyrics read far brighter than the music sounds ===')
print(df.nlargest(15, 'alignment_gap')[cols].to_string(index=False))
print()
print('=== MOST NEGATIVE gap: music sounds far brighter than the lyrics read ===')
print(df.nsmallest(15, 'alignment_gap')[cols].to_string(index=False))
print()
print('Read these by hand. If the tails do not look like the phenomenon, the')
print('formula is measuring something else and 4.9.3(A) cannot be interpreted.')

## F. Correlation with each input, and with the target

Section 4.9.3(D): *"A score correlating almost perfectly with valence or sentiment alone would indicate that it carries little information beyond one of its components, which bears directly on how any observed contribution should be interpreted."*

In [ ]:
from scipy.stats import spearmanr

targets = ['valence', 'valence_norm', 'lyric_sentiment', 'popularity']
corr = pd.DataFrame([
    {'alignment_gap vs': col,
     'pearson': float(np.corrcoef(df['alignment_gap'], df[col])[0, 1]),
     'spearman': float(spearmanr(df['alignment_gap'], df[col]).statistic)}
    for col in targets
])
print(corr.round(4).to_string(index=False))

print('\n-- for context: each input on its own vs popularity --')
for col in ['valence', 'lyric_sentiment']:
    print('  %-16s pearson %.4f' % (col, np.corrcoef(df[col], df['popularity'])[0, 1]))

print('\n-- alignment_gap vs popularity, within each genre --')
per_genre_corr = (df.groupby('genre')
                    .apply(lambda g: np.corrcoef(g['alignment_gap'], g['popularity'])[0, 1])
                    .sort_values())
print(per_genre_corr.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(gap, bins=120, color='#4a6fa5', edgecolor='none')
axes[0].axvline(0, color='k', lw=1, ls='--')
axes[0].set_xlabel('alignment_gap')
axes[0].set_ylabel('tracks')
axes[0].set_title('Alignment gap across the retained corpus')

axes[1].hexbin(df['alignment_gap'], df['popularity'], gridsize=60, cmap='Blues', mincnt=1)
axes[1].set_xlabel('alignment_gap')
axes[1].set_ylabel('popularity')
axes[1].set_title('Alignment gap vs popularity')
fig.tight_layout()
fig.savefig(mc.artifact('fig_gap_distribution.png'), dpi=150)
plt.show()

order = by_genre.index.tolist()
fig2, ax = plt.subplots(figsize=(11, 4.5))
ax.boxplot([df.loc[df['genre'] == g, 'alignment_gap'].values for g in order],
           labels=order, showfliers=False)
ax.axhline(0, color='k', lw=1, ls='--')
ax.set_ylabel('alignment_gap')
ax.set_title('Alignment gap by genre (ordered by mismatch share)')
plt.xticks(rotation=30, ha='right')
fig2.tight_layout()
fig2.savefig(mc.artifact('fig_gap_by_genre.png'), dpi=150)
plt.show()

## G. Save the summary

Everything here feeds Section 4.9.3(D) directly, and the profile block feeds Table 1.

In [ ]:
summary = {
    'profile': profile,
    'engineered_input_check': {
        'reference_cases_passed': bool(cases['matches'].all()),
        'max_corpus_deviation': deviation,
        'bounded_in_range': True,
    },
    'distribution': {
        'mean': float(gap.mean()),
        'median': float(gap.median()),
        'std': float(gap.std()),
        'share_positive_pct': float(100 * (gap > 0).mean()),
        'agreement_by_tolerance': tol.round(3).to_dict('records'),
    },
    'by_genre': by_genre.round(4).reset_index().to_dict('records'),
    'correlations': corr.round(6).to_dict('records'),
    'tau_used_for_mismatch': TAU,
}
path = mc.save_json(summary, 'behavioral_checks.json')
print('wrote', path)

---
### What to read off this notebook before going further

1. **Correlation with `lyric_sentiment`.** If it is very high, the gap is largely a rescaled sentiment score and its marginal contribution over the Control model will be small by construction. Report it either way -- Section 4.9.3(D) requires it, and it is the honest explanation for a small ΔR2.
2. **Correlation with `popularity`.** A near-zero raw correlation does not rule out a contribution (trees can use it non-linearly and in interaction), but it sets expectations for the size of the effect.
3. **Spread of the distribution.** A wide spread means the two signals are genuinely carrying different information in this corpus, which is a precondition for the feature to be able to add anything.
4. **Genre ordering.** If it broadly tracks Attia's, that is independent corroboration the formula measures the intended phenomenon, and it is reportable regardless of what the model does.

Next: `5_splits.ipynb`.